In [ ]:
# ── COLAB SETUP ───────────────────────────────────────────────────────────────
# Run this cell first on Colab to install deps, clone the repo, and fetch the
# birefnet-preprocessed dataset. No-op when running locally.
import os, sys
IS_COLAB = 'google.colab' in sys.modules
if IS_COLAB:
    REPO_URL = 'https://github.com/AlessandroGhiotto/industrial-AD2L.git'
    REPO_DIR = '/content/industrial-AD2L'
    if not os.path.exists(REPO_DIR):
        !git clone -q $REPO_URL $REPO_DIR
    %cd $REPO_DIR
    !pip install -q timm gdown
    DATA_DIR = 'dataset/adl-2025-2026-anomaly-detection_birefnet'
    if not os.path.exists(DATA_DIR):
        !gdown -q 'https://drive.google.com/uc?id=1lCqjjtKOhqoU3M5-4ktgiVlWnp6IFDYC' -O /content/dataset.zip
        !mkdir -p dataset && unzip -q /content/dataset.zip -d dataset/ && rm /content/dataset.zip
    print('Colab ready — CWD:', os.getcwd())


In [ ]:
# ── LOCAL SETUP ───────────────────────────────────────────────────────────────
import sys
import warnings
warnings.filterwarnings('ignore', message='No positive class found in y_true')
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
sys.path.append(str(Path.cwd()))

from src import config_final as cfg
from src import unet_final as unet
from src import patchcore_final as pc
from src import ensemble_final as ens
cfg.ensure_dirs()

import torch
import numpy as np
from PIL import Image
from tqdm.auto import tqdm
import pandas as pd
from collections import defaultdict

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {DEVICE}")

# Multi-Level Ensemble: U-Net + PatchCore
This notebook implements a pixel-level fusion ensemble that leverages:
- **U-Net**: Supervised learning of semantic/localized defects.
- **PatchCore**: Unsupervised memory-based normality prior.

Workflow:
1. **Normalization**: Per-class robust percentile scaling.
2. **Optimization**: Per-class weight (alpha) selection using labeled validation anomalies.
3. **Fusion**: Pixel-level combination to preserve complementary detections.

In [ ]:
# Initialize extractors/models
print("Loading models...")
pc_extractor = pc.DINOv2(cfg.PC_BACKBONE, cfg.PC_LAYERS, cfg.PC_GRID_SIZE, DEVICE).to(DEVICE).eval()

# PCA for PatchCore
_layers_str = '_'.join(map(str, cfg.PC_LAYERS))
_bank_dir = cfg.PC_BANKS_DIR / f'L{_layers_str}_pca{cfg.PC_PCA_DIM}'
pca = pc.GPUPCA(n_components=cfg.PC_PCA_DIM, device=DEVICE)
pca.mean_ = torch.from_numpy(np.load(_bank_dir / 'pca_mean.npy')).to(DEVICE)
pca.components_ = torch.from_numpy(np.load(_bank_dir / 'pca_components.npy')).to(DEVICE)

banks = {}
classes = sorted([d.name for d in cfg.DATASET_DIR.iterdir() if d.is_dir() and d.name.startswith('class_')])
for cls in classes:
    banks[cls] = torch.from_numpy(np.load(_bank_dir / f'{cls}_bank.npy')).to(DEVICE)

In [ ]:
results = []
per_class_weights = {}

PC_FEAT_DIM = pc_extractor.backbone.embed_dim  # per-layer hidden (1024 for ViT-L, 768 for ViT-B)
STRATEGY = cfg.ENSEMBLE_STRATEGY
GRID = {
    'two_tier':       cfg.ENSEMBLE_TAUS,
    'unet_first_max': cfg.ENSEMBLE_BETAS,
}.get(STRATEGY, cfg.ENSEMBLE_ALPHAS)
PARAM_NAME = {'two_tier': 'Tau', 'unet_first_max': 'Beta'}.get(STRATEGY, 'Alpha')
print(f"Fusion strategy: {STRATEGY}  (grid size: {len(GRID)})")
print("Score recipe:  UNet = raw sigmoid * FG  |  "
      f"PC = clean(erode={cfg.PC_FG_ERODE_PX}px, open={cfg.PC_OPEN_RADIUS}px) "
      f"→ excess_above_p99 → drop CCs<{cfg.PC_MIN_AREA}px")

def _raw_maps(model, img_pil, img_path, bank, use_tta):
    """Returns (unet_prob_FG, pc_raw, fg_mask)."""
    fg = ens.foreground_mask(unet, img_pil, cfg.UNET_IMAGE_SIZE)
    vid = ens.parse_view_id(img_path)
    u = ens.infer_unet_map(unet, model, img_pil, vid, DEVICE,
                           cfg.UNET_IMAGE_SIZE, use_tta=use_tta, fg_mask=fg)
    p = ens.infer_pc_map(pc, pc_extractor, pca, bank, img_path, DEVICE,
                        cfg.PC_LAYERS, cfg.PC_GRID_SIZE,
                        out_size=cfg.UNET_IMAGE_SIZE, feat_dim=PC_FEAT_DIM,
                        fg_mask=None)
    return u, p, fg

def _clean_pc(p_raw, fg):
    return ens.clean_pc_map(p_raw, fg_mask=fg,
                            erode_px=cfg.PC_FG_ERODE_PX,
                            open_radius=cfg.PC_OPEN_RADIUS)

def _pc_signal(p_raw, fg, p_p99):
    p_clean = _clean_pc(p_raw, fg)
    p_excess = ens.excess_above_p99(p_clean, p_p99)
    return ens.remove_small_blobs(p_excess, min_area=cfg.PC_MIN_AREA, threshold=0.0)

def _stratified_good(class_name, n_per_view=8):
    """Sample good images evenly across the 5 views (avoids view-1-only bias)."""
    all_paths = ens.get_class_samples(cfg.DATASET_DIR, class_name, 'train')
    by_view = defaultdict(list)
    for p in all_paths:
        by_view[ens.parse_view_id(p)].append(p)
    out = []
    for v in sorted(by_view):
        out.extend(by_view[v][:n_per_view])
    return out

# Cache per-class p99 so the viz cell can reuse it
pc_p99_per_class = {}

for cls in classes:
    print(f"\n=== Processing {cls} ===")

    unet_ckpt = cfg.UNET_MODEL_DIR / cls / 'best.pt'
    unet_model = unet.load_model(str(unet_ckpt), DEVICE)
    pc_bank = banks[cls]

    # 1. Calibration: PC p99 on CLEANED good maps (so excess_above_p99 is consistent).
    print("  Calibrating PC p99 on good images...")
    good_paths = _stratified_good(cls, n_per_view=8)
    pc_good_clean = []
    for gp in good_paths:
        img_pil = Image.open(gp).convert('RGB')
        _, p_raw, fg = _raw_maps(unet_model, img_pil, gp, pc_bank, use_tta=False)
        pc_good_clean.append(_clean_pc(p_raw, fg))
    _, p_p99 = ens.compute_calibration_stats(pc_good_clean)
    pc_p99_per_class[cls] = p_p99
    print(f"  PC p99 (good, post-clean): {p_p99:.4f}")

    # 2. Per-class parameter optimization on ALL labeled anomalies
    print(f"  Optimizing {PARAM_NAME}...")
    anomaly_root = cfg.DATASET_DIR / cls / 'train'
    val_anom_paths = []
    for d in anomaly_root.iterdir():
        if d.is_dir() and d.name.startswith('anomaly_'):
            val_anom_paths.extend(list(d.glob('*.png')))
    print(f"  Using {len(val_anom_paths)} labeled anomalies for grid search.")

    if val_anom_paths:
        u_val, p_val, gt_val = [], [], []
        for ap in tqdm(val_anom_paths, desc=f"Opt {cls}"):
            mask = ens.load_mask_for_image(ap, cfg.DATASET_DIR,
                                           size=(cfg.UNET_IMAGE_SIZE, cfg.UNET_IMAGE_SIZE))
            if mask is None or mask.sum() == 0:
                continue
            img_pil = Image.open(ap).convert('RGB')
            u_raw, p_raw, fg = _raw_maps(unet_model, img_pil, ap, pc_bank, use_tta=cfg.UNET_USE_TTA)
            u_val.append(u_raw)
            p_val.append(_pc_signal(p_raw, fg, p_p99))
            gt_val.append(mask)

        best_w, best_ap = ens.optimize_weights_per_class(
            cls, u_val, p_val, gt_val, GRID,
            strategy=STRATEGY, verbose=True)

        # Sanity: pure-UNet AP on same pooled pixels
        gt_flat = np.concatenate([g.ravel().astype(np.uint8) for g in gt_val])
        unet_flat = np.concatenate([u.ravel() for u in u_val])
        from sklearn.metrics import average_precision_score
        unet_only_ap = average_precision_score(gt_flat, unet_flat)
        print(f"  Best {PARAM_NAME}: {best_w:.3f} (AP {best_ap:.4f})  |  "
              f"UNet-only AP: {unet_only_ap:.4f}  |  gain: {best_ap - unet_only_ap:+.4f}")
    else:
        best_w = 0.0
        print(f"  No labels, using default param={best_w}")

    per_class_weights[cls] = best_w

    # 3. Test inference
    test_paths = ens.get_class_samples(cfg.DATASET_DIR, cls, 'test')
    for tp in tqdm(test_paths, desc=f"Infer {cls}"):
        img_pil = Image.open(tp).convert('RGB')
        orig_size = img_pil.size  # (W, H)
        u_raw, p_raw, fg = _raw_maps(unet_model, img_pil, tp, pc_bank, use_tta=cfg.UNET_USE_TTA)
        p_signal = _pc_signal(p_raw, fg, p_p99)

        fused = ens.fuse_maps(u_raw, p_signal, best_w, strategy=STRATEGY)
        fused_pil = Image.fromarray((fused * 255).astype(np.uint8)).resize(orig_size, Image.BILINEAR)
        fused_orig = np.asarray(fused_pil, dtype=np.float32) / 255.0
        results.append({'ID': tp.stem, 'Label': ens.rle_encode_q8(fused_orig)})

print("\nPer-class chosen weights:")
for c, a in per_class_weights.items():
    print(f"  {c}: {a:.3f}")


In [ ]:
from src import visualize_final as viz
from sklearn.metrics import average_precision_score
import torch
from PIL import Image
import numpy as np
from tqdm.auto import tqdm

N_VIS = 10
viz_outputs = []

for cls in classes:
    print(f"Visualizing {cls}...")
    unet_ckpt = cfg.UNET_MODEL_DIR / cls / 'best.pt'
    unet_model = unet.load_model(str(unet_ckpt), DEVICE)
    pc_bank = banks[cls]
    p_p99 = pc_p99_per_class[cls]

    anomaly_root = cfg.DATASET_DIR / cls / 'train'
    val_anom_paths = []
    for d in anomaly_root.iterdir():
        if d.is_dir() and d.name.startswith('anomaly_'):
            val_anom_paths.extend(list(d.glob('*.png'))[:1])

    for ap in tqdm(val_anom_paths, desc=f"Viz {cls}"):
        mask = viz.load_mask(
            str(cfg.DATASET_DIR / cls / 'ground_truth_train' / ap.parent.name / ap.name),
            size=(cfg.UNET_IMAGE_SIZE, cfg.UNET_IMAGE_SIZE))
        img_pil = Image.open(ap).convert('RGB')
        u_raw, p_raw, fg = _raw_maps(unet_model, img_pil, ap, pc_bank, use_tta=cfg.UNET_USE_TTA)
        p_signal = _pc_signal(p_raw, fg, p_p99)

        best_w = per_class_weights[cls]
        fused = ens.fuse_maps(u_raw, p_signal, best_w, strategy=STRATEGY)
        if mask.shape != fused.shape:
            mask = np.array(Image.fromarray(mask).resize(
                (fused.shape[1], fused.shape[0]), Image.NEAREST))
        ap_score = average_precision_score(mask.ravel().astype(int), fused.ravel())

        viz_outputs.append({
            'input': img_pil,
            'gt': mask,
            'heatmap': fused,
            'class_name': cls,
            'anomaly_name': ap.parent.name,
            'ap': ap_score,
        })

pdf_path = cfg.OUTPUT_DIR / 'ensemble_visualizations.pdf'
viz.export_to_pdf(viz_outputs, str(pdf_path), title_prefix="Ensemble Results:")

In [ ]:
sub_df = pd.DataFrame(results).sort_values('ID').reset_index(drop=True)
sub_path = cfg.OUTPUT_DIR / 'submission_ensemble_final.csv'
sub_df.to_csv(sub_path, index=False)
print(f"Ensemble submission saved to: {sub_path}")